# 03 — Visualizar enero 2016 y exportar parquet

ERA5-Land acumula `tp` y `ssrd` desde las 00 UTC; a las 00 UTC viene el total de 24 h, no de una hora. El extractor desacumula: `tp_mm` es mm de esa hora y `ssrd_wm2` es el promedio de esa hora en W/m². El agregado diario suma precip y promedia el resto. La primera 00 UTC queda en NaN.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from ecmwf_ml.client import target_path
from ecmwf_ml.config import get_settings
from ecmwf_ml.extract import daily_frame, to_frame, write_parquet

%matplotlib inline

settings = get_settings()
nc_path = target_path(settings.month_start, settings.month_end, settings)
assert nc_path.exists(), f"No está {nc_path}. Corré primero notebooks/02_download_month.ipynb."
nc_path

In [ ]:
hourly = to_frame(nc_path, settings)
daily = daily_frame(hourly)
print(hourly.dtypes)
print(f"horas={len(hourly)} días={len(daily)}")
hourly.describe(include="number").T

In [ ]:
hourly.head(8)

## Series horarias y diarias

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False)
h = hourly.sort_values("time")
d = daily.sort_values("date")

axes[0, 0].plot(h["time"], h["t2m_c"], lw=0.8)
axes[0, 0].set_ylabel("2 m temperature (°C)")
axes[0, 1].plot(h["time"], h["wind_speed"], lw=0.8)
axes[0, 1].set_ylabel("10 m wind speed (m/s)")
axes[1, 0].bar(d["date"], d["tp_mm"], width=0.8)
axes[1, 0].set_ylabel("Daily precip (mm)")
axes[1, 1].plot(h["time"], h["ssrd_wm2"], lw=0.8)
axes[1, 1].set_ylabel("Hourly SSRD (W/m²)")
axes[1, 0].tick_params(axis="x", rotation=30)
axes[1, 1].tick_params(axis="x", rotation=30)
fig.suptitle("ERA5-Land — enero 2016 — punto Chile")
fig.tight_layout()
plt.show()

## Radiación horaria (W/m²)

`ssrd_wm2` es el flujo medio de la hora que termina en cada timestamp. En enero el pico cae cerca de 16–18 UTC (mediodía solar). De noche debe ir a ~0, no quedarse en un valor constante.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharey=True)
h = hourly.sort_values("time")
d = daily.sort_values("date")
first_days = h[h["time"] < h["time"].min() + pd.Timedelta(days=3)]

axes[0].plot(h["time"], h["ssrd_wm2"], lw=0.8)
axes[0].set_ylabel("W/m²")
axes[0].set_title("SSRD horario — mes completo")
axes[0].set_ylim(0, 1200)

axes[1].plot(first_days["time"], first_days["ssrd_wm2"], lw=1.0, marker="o", ms=3)
axes[1].set_ylabel("W/m²")
axes[1].set_title("SSRD horario — primeros 3 días (ciclo diurno)")
axes[1].set_ylim(0, 1200)
axes[1].tick_params(axis="x", rotation=30)

fig.suptitle("Radiación solar incidente — W/m²")
fig.tight_layout()
plt.show()

print(
    f"ssrd_wm2 min={h['ssrd_wm2'].min():.2f}  "
    f"max={h['ssrd_wm2'].max():.1f}  "
    f"media diaria (promedio de horas)={d['ssrd_wm2'].mean():.1f}"
)

## Viento horario

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
h = hourly.sort_values("time")
axes[0].plot(h["time"], h["u10"], label="u10")
axes[0].plot(h["time"], h["v10"], label="v10")
axes[0].set_ylabel("m/s")
axes[0].legend()
axes[1].scatter(h["time"], h["wind_dir_from"], s=4, alpha=0.5)
axes[1].set_ylabel("dir from (°)")
axes[1].set_ylim(0, 360)
axes[1].tick_params(axis="x", rotation=30)
fig.suptitle("Viento a 10 m (horario)")
fig.tight_layout()
plt.show()

## Sanity checks para ML

In [ ]:
checks = {
    "sin nulos en t2m_c": int(hourly["t2m_c"].isna().sum()),
    "tp_mm >= -1e-6": bool((hourly["tp_mm"].fillna(0) >= -1e-6).all()),
    "ssrd_wm2 >= -1": bool((hourly["ssrd_wm2"].fillna(0) >= -1).all()),
    "ssrd_wm2 pico < 1400": bool((hourly["ssrd_wm2"].dropna() < 1400).all()),
    "t2m_c rango razonable": bool(hourly["t2m_c"].between(-20, 45).all()),
}
display(pd.Series(checks, name="check"))
daily.describe().T

## Export

In [ ]:
out = write_parquet(hourly, settings=settings)
print(out)
print(f"{out.stat().st_size / 1024:.1f} KiB")
pd.read_parquet(out).head()